Sentiment Analysis Methodoloygy
 
The dataset where the models were tested is available here: https://huggingface.co/datasets/ElKulako/stocktwits-crypto
It contains all cryptocurrency-related posts from the StockTwits website (around 1.3M), from 1st of November 2021 to the 15th of June 2022.
 
The provided data was already cleaned and preprocessed, in particular it was removed:
 
- cashtags, hashtags, usernames,
- URLs, crypto wallets,
- Chinese, Korean and Japanese characters,
- (most) UTF-8 encoding issues
- All posts shorter than 4 words
- All duplicate posts
- Spacing and punctuation issues, converted all text to lowercase

Initially, i intended to test on the dataset the following models:
 
- Vader: Lexicon and rule-based sentiment analysis tool that is specifically attuned to sentiments expressed in social media
- Support Vector Machines: A traditional, statistical classifier.
- Random Forest: An ensemble machine learning method utilizing deep decision trees.
- Gemma 2B: Google's lightweight, 2-billion parameter Large Language Model.

All this models were to be benchmarked against CryptoBERT, a specialized Transformer machine learning model trained on 3.2 millions social media posts of various cryptocurrencies. 
The benchmark results for CryptoBERT are publicly available, making it an ideal competitor, especially since it was explicitly fine-tuned on a portion of this exact StockTwits dataset.
 
After some testing i decided to drop Gemma 2B and Random forest for two main reasons:
 
- Performance: Preliminary testing showed they did not maintain competitive accuracy compared to the SVM and CryptoBERT.
- Computational Efficiency: At scale the inference time for the LLM and the training complexity of the Random Forest were prohibitively high.

Consequently, the final evaluation focuses on a comparison between VADER, SVM and CryptoBERT.
 
Results & Evaluation
 
VADER and SVM were evaluated on the same exact 83k rows test set of the CryptoBERT paper
 
VADER: Accuracy: 37.34% | Macro F1: 0.3679
 
As expected, the traditional rule-based lexicon struggled. 
VADER's reliance on standard English definitions proved inadequate for capturing the nuanced, slang-heavy and emoji-driven nature of cryptocurrency posts.
 
CryptoBERT: Accuracy: 55.60% | Macro F1: 0.5579
 
The specialized transformer established a balanced standard. 
Thanks to its deep contextual embeddings pre-trained on crypto-specific text, it successfully managed the complex class imbalances and set a high benchmark for the Macro F1 score.
 
SVM: Accuracy: 59.71% | Macro F1: 0.5916
 
The SVM performed exceptionally well, outperforming CryptoBERT in both categories. 
By utilizing a wide range of n-grams (1–5), the model successfully captured the specific "power phrases" and slang that drive sentiment on StockTwits. 
It achieved an accuracy 4.11% higher than the state-of-the-art transformer.
 
Conclusion

The results demonstrate that a properly tuned statistical machine learning pipeline can surpass a specialized Transformer model in both raw accuracy and Macro F1 score on this dataset.
The SVM’s superior performance, combined with its significantly lower computational overhead, makes it the most viable and scalable solution for high-frequency sentiment analysis in the cryptocurrency domain.

In [1]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
import pickle

TARGET_NAMES = ["Bearish", "Neutral", "Bullish"]
LABEL_ORDER  = [0, 1, 2]

vader = SentimentIntensityAnalyzer()

def get_vader_sentiment(text: str) -> int:
    compound = vader.polarity_scores(text)["compound"]
    if compound >= 0.05:
        return 2   # Bullish
    elif compound <= -0.05:
        return 0   # Bearish
    else:
        return 1   # Neutral

def evaluate_model(model_name: str, y_true, y_pred):
    
    print(f" {model_name}")
    print(f"  Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred, labels=LABEL_ORDER, target_names=TARGET_NAMES, digits=4))

if __name__ == "__main__":

    df_train_1 = pd.read_csv("../data/sentiment/stockwits_train_1.csv")
    df_train_2 = pd.read_csv("../data/sentiment/stockwits_train_2.csv")
    df_test = pd.read_csv("../data/sentiment/stockwits_test.csv")

    df_train = pd.concat([df_train_1, df_train_2], ignore_index=True)
    
    x_train, y_train = df_train["text"].tolist(), df_train["label"].tolist()
    x_test, y_test = df_test["text"].tolist(), df_test["label"].tolist()
    
    results_df = pd.DataFrame({"text": x_test, "label": y_test})

    print("Running VADER...")
    results_df["vader_pred"] = results_df["text"].apply(get_vader_sentiment)
    
    print("Training & running SVM...")
    svm_pipeline = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 5),  min_df=1, max_df=0.95,)),
                             ("clf", LinearSVC(class_weight="balanced", dual=False, max_iter=2000)),])
    
    svm_pipeline.fit(x_train, y_train)
    results_df["svm_pred"] = svm_pipeline.predict(x_test)

    evaluate_model("VADER", results_df["label"], results_df["vader_pred"])
    evaluate_model("SVM", results_df["label"], results_df["svm_pred"])
    
    print(f"CryptoBert")
    print(f"  Accuracy: 0.5560")
    print(f"  Macro AVG F1 Score: 0.5579")
    print(f"  Macro AVG Precision: 0.4658")
    print(f"  Macro AVG Recall: 0.6044")

    pickle.dump(svm_pipeline, open("../data/models/svm.pkl", "wb"))

Running VADER...
Training & running SVM...
 VADER
  Accuracy: 0.3734
              precision    recall  f1-score   support

     Bearish     0.3097    0.3455    0.3266     20927
     Neutral     0.4633    0.3623    0.4066     37758
     Bullish     0.3351    0.4144    0.3706     24572

    accuracy                         0.3734     83257
   macro avg     0.3694    0.3741    0.3679     83257
weighted avg     0.3869    0.3734    0.3759     83257

 SVM
  Accuracy: 0.5971
              precision    recall  f1-score   support

     Bearish     0.5956    0.5068    0.5476     20927
     Neutral     0.6530    0.5491    0.5966     37758
     Bullish     0.5451    0.7477    0.6305     24572

    accuracy                         0.5971     83257
   macro avg     0.5979    0.6012    0.5916     83257
weighted avg     0.6067    0.5971    0.5943     83257

CryptoBert
  Accuracy: 0.5560
  Macro AVG F1 Score: 0.5579
  Macro AVG Precision: 0.4658
  Macro AVG Recall: 0.6044
